### ロード

In [26]:
import pandas as pd
import numpy as np
from scipy.stats import skew
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [27]:
# --- ファイル読み込み ---

df_raw = pd.read_parquet("liq_engine_raw.parquet", engine="pyarrow")
df_feats_d = pd.read_parquet("liq_engine_feats_d.parquet", engine="pyarrow")
df_feats_w = pd.read_parquet("liq_engine_feats_w.parquet", engine="pyarrow")
df_feats_w.columns

Index(['^GSPC', 'RRPONTSYD', 'DTB3', 'DX-Y.NYB', 'HG=F', 'GC=F', 'DFII10',
       'T10YIE', 'WALCL', 'WDTGAL', 'WCURCIR', 'NFCI', 'STLFSI4', 'ANFCI',
       'NDFACBM027SBOG', 'CPF3M', 'PERMIT', 'RRP_filled', 'Net_Liquidity',
       'Net_Liquidity_z52', 'NFCI_z52', 'Liq_eff', 'CPF3M_DTB3_Spread',
       'CPF3M_DTB3_Spread_z52', 'NDFACBM027SBOG_z52', 'STLFSI4_z52',
       'Cu_Au_Ratio', 'Cu_Au_Ratio_z52', 'PERMIT_GOLD_ratio',
       'PERMIT_GOLD_ratio_z52', 'DXY_z52', 'DFII10_z52', 'ANFCI_z52'],
      dtype='str')

### 特徴量性格診断

In [28]:
# 特徴量生成
df_feats_w["DFII10_roc4"] = df_feats_w["DFII10"].diff(4)

In [29]:
# リターン生成
lag_weeks = 0
forward_weeks = 8
target_return = f'lag_{lag_weeks}weeks_return_{forward_weeks}weeks'
df_feats_w[target_return] = (
    (df_feats_w["^GSPC"].shift(-(lag_weeks + forward_weeks))) / 
     df_feats_w["^GSPC"].shift(-lag_weeks) -1
)

In [30]:
# stats可視化 median IQR cvar_5
feature = "DXY_z52"
is_inverse = False

df = df_feats_w[[feature, target_return]].copy()
df = df.loc["2010-01-01":"2016-01-01"]
#df = df.loc["2016-01-01":"2020-01-01"]
#df = df.loc["2020-01-01":"2024-01-01"]
#df = df.loc["2024-01-01":"2026-01-01"]
# ランクの作成
df[f'{feature}_rank'] = pd.qcut(df[feature].dropna(), 5, labels=False, duplicates='drop') + 1
if is_inverse:
    df[f'{feature}_rank'] = 6 - df[f'{feature}_rank'] # 1が最悪(重力)、5が最高(浮力)に統一

# クロス集計を作成（行：現在、列：次期）
df['next_rank'] = df[f'{feature}_rank'].shift(-1)
transition_matrix = pd.crosstab(
    df[f'{feature}_rank'], 
    df['next_rank'], 
    normalize='index'
)
# 各統計量の計算
stats = []
for r in range(1, 6):
    d = df[df[f'{feature}_rank'] == r][target_return]
    stats.append({
        'rank': r,
        'median': d.median(),
        'iqr': d.quantile(0.75) - d.quantile(0.25),
        'cvar_5': d[d <= d.quantile(0.05)].mean(),
        'win_rate': (d > 0).mean()
    })
s_df = pd.DataFrame(stats)

fig = make_subplots(
    rows=2, cols=3, 
    subplot_titles=('IQR: Lower is better', 'Median', 'Tail Risk (CVaR 5%)', "Win_Rate", 'Transition Probability (Next Week)'),
    vertical_spacing=0.1,
    specs=[[{"type": "bar"}, {"type": "scatter"}, {"type": "bar"}],
           [{"type": "scatter"}, {"type": "heatmap", "colspan": 2}, None]] # ヒートマップを2列分使う
)

fig.add_trace(go.Bar(x=s_df['rank'], y=s_df['iqr'], name='IQR', marker_color='lightblue'), row=1, col=1)
fig.add_trace(go.Scatter(x=s_df['rank'], y=s_df['median'], name='Median', mode='lines+markers', line_color='orange'), row=1, col=2)
fig.add_trace(go.Bar(x=s_df['rank'], y=s_df['cvar_5'], name='CVaR', marker_color='crimson'), row=1, col=3)
fig.add_trace(go.Scatter(x=s_df['rank'], y=s_df['win_rate'], name='Win Rate', marker_color='crimson'), row=2, col=1)
# 遷移ヒートマップの追加
fig.add_trace(
    go.Heatmap(
        z=transition_matrix.values,
        x=[f"Next R{i}" for i in transition_matrix.columns],
        y=[f"Now R{i}" for i in transition_matrix.index],
        colorscale='Viridis',
        text=np.around(transition_matrix.values, 2), # 数値を表示
        texttemplate="%{text}",
        showscale=False
    ),
    row=2, col=2
)
fig.update_layout(
    title_text=f"Personality Profile: {feature} - Lag:{lag_weeks} weeks, Return:forward {forward_weeks} weeks",
    showlegend=False, template="plotly_dark"
)
fig.show(renderer="browser")


In [31]:
# 箱ひげ図の描画
plot_df = df.dropna(subset=[target_return, f'{feature}_rank'])

rank_name = f'{feature}_rank'
fig = px.box(
    plot_df,
    x=rank_name,
    y=target_return,
    color=rank_name,
    points="all", # 全データ点（ジッター）を表示して密度を確認
    title=f"Market Physics: S&P 500 {forward_weeks}W Forward Return by {feature}_rank",
    labels={target_return: f'{forward_weeks}-Week Forward Return', '{feature}_rank': 'Liquidity Regime'},
    category_orders={f"{feature}_rank": ['1: Very Low (Heavy)', '2: Low', '3: Neutral', '4: High', '5: Very High (Buoyancy)']}
)

# 0%のライン（損益分岐）を強調
fig.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Break Even")

# レイアウト調整
fig.update_layout(
    xaxis_title="Liquidity Regime (Liq_eff Z-score)",
    yaxis_title=f"Forward {forward_weeks}W Return (%)",
    showlegend=False,
    template="plotly_white",
    yaxis=dict(tickformat=".1%")
)

# 表示
fig.show(renderer="browser")

### マクロ潮流
2011～2015：  
・実質金利が沈み、クレジットギャップが底を打ちDSRも低下  
・借金返済に追われているため中央銀行がマイナス金利で無重力状態をつくる  
・流動性を入れても経済はまわりにくいが株価は浮く  
2016～2019：  
・期待インフレが安定。実質金利も戻っていく。DSRはゆっくりと上昇  
・インフレなき成長。重量が徐々に正常化。適温  
・ゴールデンレジーム。Liq_effの安定性に注目  
2020～2023：  
・コロナショックで実質金利と期待インフレが底。2022年に向けて歴史的な打ち上げ。DSR急上昇  
・インフレの牙、猛烈な金利引き上げによるショック  
・最も厳しい過渡期扱い。  
2024～2026：  
・実質金利と期待インフレが高い状態で横ばい。DSRがピークアウト  
・ショック後だが重力は戻らない。高い重力市場が適応  
・重量が強いため流動性だけでなく強い体温がないと浮力が相殺される


In [ ]:
df = df_raw[["credit_gap", "dsr", "DFII10", "T10YIE", "WALCL"]].dropna(how="all")
def zscore(df):
    return (df - df.mean()) / df.std()

credit_gap = df["credit_gap"].dropna()
#credit_gap = zscore(credit_gap)
dsr = df["dsr"].dropna().resample("ME").ffill().dropna()
#dsr = zscore(dsr)
DFII10 = df["DFII10"].dropna().resample("ME").mean().dropna().rolling(6).mean().dropna()
T10YIE = df["T10YIE"].dropna().resample("ME").mean().dropna().rolling(6).mean().dropna()
#T10YIE = zscore(T10YIE)
WALCL = df["WALCL"].dropna()
#WALCL = zscore(WALCL)
T10YIE

2003-06-30    1.762978
2003-07-31    1.781582
2003-08-31    1.816937
2003-09-30    1.854477
2003-10-31    1.927698
2003-11-30    2.040225
2003-12-31    2.152177
2004-01-31    2.218358
2004-02-29    2.251249
2004-03-31    2.296784
2004-04-30    2.335229
2004-05-31    2.383460
2004-06-30    2.434048
2004-07-31    2.469183
2004-08-31    2.485802
2004-09-30    2.480029
2004-10-31    2.467236
2004-11-30    2.449319
2004-12-31    2.445095
2005-01-31    2.449126
2005-02-28    2.467858
2005-03-31    2.531823
2005-04-30    2.576363
2005-05-31    2.571855
2005-06-30    2.532082
2005-07-31    2.499249
2005-08-31    2.472460
2005-09-30    2.437384
2005-10-31    2.418345
2005-11-30    2.415102
2005-12-31    2.418465
2006-01-31    2.436631
2006-02-28    2.461052
2006-03-31    2.465472
2006-04-30    2.475568
2006-05-31    2.507667
2006-06-30    2.546502
2006-07-31    2.574918
2006-08-31    2.585860
2006-09-30    2.564690
2006-10-31    2.520697
2006-11-30    2.460570
2006-12-31    2.416767
2007-01-31 

In [ ]:
fig = go.Figure()
#fig.add_trace(go.Scatter(x=credit_gap.index, y=credit_gap, mode='lines', name="credit_gap" ))
fig.add_trace(go.Scatter(x=dsr.index, y=dsr, mode='lines', name="dsr" ))
#fig.add_trace(go.Scatter(x=DFII10.index, y=DFII10, mode='lines', name="DFII10" ))
#fig.add_trace(go.Scatter(x=T10YIE.index, y=T10YIE, mode='lines', name="T10YIE" ))
#fig.add_trace(go.Scatter(x=WALCL.index, y=WALCL, mode='lines', name="WALCL" ))

fig.show(renderer="browser")

pd.set_option("display.max.rows", None)
print(df)


            credit_gap   dsr  DFII10  T10YIE      WALCL
1996-07-01        -3.6   NaN     NaN     NaN        NaN
1996-10-01        -3.9   NaN     NaN     NaN        NaN
1997-01-01        -4.2   NaN     NaN     NaN        NaN
1997-04-01        -3.7   NaN     NaN     NaN        NaN
1997-07-01        -3.0   NaN     NaN     NaN        NaN
1997-10-01        -2.6   NaN     NaN     NaN        NaN
1998-01-01        -1.9   NaN     NaN     NaN        NaN
1998-04-01        -0.1   NaN     NaN     NaN        NaN
1998-07-01         0.7   NaN     NaN     NaN        NaN
1998-10-01         1.6   NaN     NaN     NaN        NaN
1999-01-01         2.2   NaN     NaN     NaN        NaN
1999-04-01         2.7   NaN     NaN     NaN        NaN
1999-07-01         3.9   NaN     NaN     NaN        NaN
1999-10-01         4.2   NaN     NaN     NaN        NaN
2000-01-01         4.6   NaN     NaN     NaN        NaN
2000-04-01         4.8   NaN     NaN     NaN        NaN
2000-07-01         4.8   NaN     NaN     NaN    

### 

### 